In [6]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from sklearn.ensemble import BaggingClassifier,BaggingRegressor
from sklearn.metrics import log_loss,accuracy_score,precision_score,f1_score,mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer,make_column_selector

# Bagging Classifier

In [8]:
hr=pd.read_csv("D:\Machine Learning\Datasets\HR_comma_sep.csv")

X , y = hr.drop('left',axis=1) , hr['left']
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
col_transformer = ColumnTransformer([("OHE", ohe, make_column_selector(dtype_include=object))], 
                                    remainder='passthrough', 
                                    verbose_feature_names_out=False)
col_transformer = col_transformer.set_output(transform="pandas")
X = col_transformer.fit_transform(X)

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)


dtc1=DecisionTreeClassifier(random_state=25,max_depth=3)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=2)
lr=LogisticRegression()


lr

In [16]:
bagging=BaggingClassifier(random_state=25,estimator=lr,n_estimators=15)
bagging.fit(X_train,y_train)
y_pred=bagging.predict(X_test)
print('accuracy_score: ',accuracy_score(y_test,y_pred))


accuracy_score:  0.8059568793065126


dtc1

In [17]:
bagging=BaggingClassifier(random_state=25,estimator=dtc1,n_estimators=15)
bagging.fit(X_train,y_train)
y_pred=bagging.predict(X_test)
print('accuracy_score: ',accuracy_score(y_test,y_pred))

accuracy_score:  0.9564347632807291


dtc2

In [18]:
bagging=BaggingClassifier(random_state=25,estimator=dtc2,n_estimators=15)
bagging.fit(X_train,y_train)
y_pred=bagging.predict(X_test)
print('accuracy_score: ',accuracy_score(y_test,y_pred))

accuracy_score:  0.8501889308735274


In [22]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


lr=LogisticRegression()
scaler=StandardScaler()
knn=KNeighborsClassifier(n_neighbors=5)
pipe_knn=Pipeline([('SCL',scaler),('KNN',knn)])
svm=SVC(kernel='rbf')
pipe_svm=Pipeline([('SCL',scaler),('SVM',svm)])

dtc1=DecisionTreeClassifier(random_state=25,max_depth=3)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=2)
lr=LogisticRegression()


In [28]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)
esitimators=[lr,pipe_knn,pipe_svm,dtc1,dtc2]
scores=[]
for e in esitimators:
    bagg=BaggingClassifier(random_state=25,estimator=e,n_estimators=15)
    bagg.fit(X_train,y_train)
    y_pred=bagg.predict(X_test)
    scores.append([e,accuracy_score(y_test,y_pred)])
scores_df=pd.DataFrame(scores,columns=('Estimators','score'))
scores_df.sort_values('score',ascending=False)

,Estimators,score
3,"DecisionTreeClassifier(max_depth=3, random_sta...",0.956435
2,"(StandardScaler(), SVC())",0.951323
1,"(StandardScaler(), KNeighborsClassifier())",0.944877
4,"DecisionTreeClassifier(max_depth=2, random_sta...",0.850189
0,LogisticRegression(),0.805957


# Bagging Regressor

In [38]:
boston=pd.read_csv("D:\Machine Learning\Datasets\Boston.csv")
X,y=boston.drop('medv',axis=1),boston['medv']

In [43]:
from sklearn.linear_model import LinearRegression

from sklearn.neighbors import KNeighborsRegressor

scaler=StandardScaler()
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=25)


dtc1=DecisionTreeRegressor(random_state=25,max_depth=3)
dtc2=DecisionTreeRegressor(random_state=25,max_depth=2)
lr=LinearRegression()
knn=KNeighborsRegressor(n_neighbors=5)
pipe_knn=Pipeline([('SCL',scaler),('KNN',knn)])

In [46]:

esitimators=[lr,pipe_knn,dtc1,dtc2]
scores=[]
for e in esitimators:
    bagg=BaggingRegressor(random_state=25,estimator=e,n_estimators=15)
    bagg.fit(X_train,y_train)
    y_pred=bagg.predict(X_test)
    scores.append([e,mean_absolute_error(y_test,y_pred)])
scores_df=pd.DataFrame(scores,columns=('Estimators','score'))
scores_df.sort_values('score',ascending=True)

,Estimators,score
1,"(StandardScaler(), KNeighborsRegressor())",2.722079
2,"DecisionTreeRegressor(max_depth=3, random_stat...",2.881208
0,LinearRegression(),3.159748
3,"DecisionTreeRegressor(max_depth=2, random_stat...",3.300542
